# 크래시 진단 노트북

각 셀을 **하나씩 순서대로** 실행. 어느 셀에서 커널이 죽는지가 원인.

| 죽는 셀 | 원인 | 해결 |
|---|---|---|
| 셀 2 (torch GPU) | CUDA 드라이버/torch 버전 불일치 | torch 재설치 |
| 셀 3 (FlagEmbedding import) | 의존 패키지 충돌 (transformers/peft) | FlagEmbedding 재설치 |
| 셀 4 (BGE-M3 GPU 로드) | GPU OOM 또는 CUDA kernel 충돌 | CPU 시도 또는 fp16 끄기 |
| 셀 5 (첫 encode) | GPU inference 충돌 | 배치 1 로 줄임 |
| 셀 6 (chromadb import) | sqlite/native lib 충돌 | chromadb 재설치 |

## 셀 1 — Python / 패키지 버전 확인 (안전)

In [ ]:
import sys
print(f'Python   : {sys.version.split()[0]}')
print(f'Executable: {sys.executable}')

# 주요 패키지 버전
import importlib
for pkg in ['torch','transformers','tokenizers','FlagEmbedding','chromadb','sentencepiece','numpy']:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', 'unknown')
        print(f'  {pkg:<18}: {ver}')
    except ImportError:
        print(f'  {pkg:<18}: NOT INSTALLED')
    except Exception as e:
        print(f'  {pkg:<18}: ERR ({e})')

## 셀 2 — PyTorch CUDA 동작 확인

GPU 텐서 생성 + 간단 연산. 여기서 죽으면 **CUDA 드라이버 vs torch 빌드 버전 불일치**.

In [ ]:
import torch
print(f'torch       : {torch.__version__}')
print(f'CUDA build  : {torch.version.cuda}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total  : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
    print(f'VRAM used   : {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

    # 작은 GPU 연산 테스트
    x = torch.randn(1000, 1000, device='cuda')
    y = x @ x.T
    print(f'\n✓ GPU 연산 OK  (결과 shape {y.shape})')
    del x, y
    torch.cuda.empty_cache()
else:
    print('⚠ CUDA 미감지 — CPU 모드로 진행해야 함')

## 셀 3 — FlagEmbedding 임포트만 (모델 로드 X)

임포트만 한다. 여기서 죽으면 **FlagEmbedding 의존성 충돌** (transformers/peft 등).

In [ ]:
from FlagEmbedding import BGEM3FlagModel
print('✓ FlagEmbedding 임포트 OK')

## 셀 4 — BGE-M3 모델 로드 (CPU 먼저)

**일부러 CPU 로 먼저** 로드해서 GPU 충돌 가능성 배제. 여기서 죽으면 **모델 자체 또는 transformers 충돌**.

In [ ]:
# CPU 로드 — GPU 충돌 가능성 배제
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''   # GPU 강제 비활성화

from FlagEmbedding import BGEM3FlagModel
print('CPU 로 BGE-M3 로드 중... (1~3분, 최초면 다운로드 ~2.3GB)')
model_cpu = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, devices=['cpu'])
print('✓ CPU 로드 성공')

## 셀 5 — CPU 로 1개 문장 인코딩 (느리지만 안전 확인)

In [ ]:
out = model_cpu.encode(
    ['삼양식품의 사업부문은 면스낵, 뉴트리션, 소스조미소재, 냉동으로 구성됩니다.'],
    return_dense=True, return_sparse=False, return_colbert_vecs=False,
)
print(f'✓ CPU 인코딩 성공 — shape {out["dense_vecs"].shape}')
print(f'  첫 5개 값: {out["dense_vecs"][0][:5]}')

## 셀 6 — chromadb 임포트 + 임시 컬렉션

여기서 죽으면 chromadb 네이티브 라이브러리 문제.

In [ ]:
import chromadb
print(f'chromadb 버전: {chromadb.__version__}')

# 메모리 클라이언트로 간단 테스트 (디스크 IO 배제)
client = chromadb.EphemeralClient()
coll = client.create_collection('test')
coll.add(
    ids=['a','b'],
    documents=['hello','world'],
    embeddings=[[0.1]*1024, [0.2]*1024],
)
print(f'✓ chromadb 메모리 컬렉션 OK ({coll.count()}개)')

## 셀 7 — GPU 모델 로드 재시도 (커널 재시작 후 진행)

**중요**: 셀 4 에서 GPU 비활성화했으므로 이 셀 실행 전 **VS Code 에서 커널 재시작** 필요.  
(Kernel → Restart Kernel)

In [ ]:
# 커널 재시작 후 처음부터 실행
import torch
from FlagEmbedding import BGEM3FlagModel

print(f'CUDA available: {torch.cuda.is_available()}')
print('GPU 로 BGE-M3 로드 중...')

# FP16 끄고 시도 (FP16 이 일부 GPU 에서 충돌 유발)
model_gpu = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, devices=['cuda:0'])
print('✓ GPU 로드 성공 (fp16=False)')

# 1개 인코딩 테스트
out = model_gpu.encode(['테스트 문장입니다.'], return_dense=True,
                       return_sparse=False, return_colbert_vecs=False)
print(f'✓ GPU 인코딩 성공 — shape {out["dense_vecs"].shape}')